# muon/gamma 抽样 → DE + CEvNS 仿真（仿照 run_20260829_background_cevns.py）

本 notebook 由 `20260730.ipynb` 改写而来，**不再直接导入 muon/gamma 文件作为模拟输入**，
而是仿照 `scripts/run_20260730_background.py`（即 `run_20260829_background_cevns.py`）的抽样逻辑：

1. 分别导入指定数量的 `muon_events` / `gamma_events` 文件作为**抽样样本**
   （例如每次 1 个 muon_events 文件 + 6 个 gamma_events 文件）。
2. 指定**模拟总时长**与 **muon/gamma 事件率**（例如 10000 s、muon 6 Hz、gamma 1 Hz）。
3. 按 `eventId` 从对应文件中**随机抽取事件**（优先无放回；候选不足时有放回补足，
   重复抽到的事件分配全新 eventId），直至 muon 数达到 `sim_time × muon_rate`、
   gamma 数达到 `sim_time × gamma_rate`。
4. 将抽样得到的 muon 与 gamma **合并**，合并后 `eventId` 从 0 重新排序。
5. 用该合并数组替代原导入文件，作为 **DE** 与 **CEvNS** 两个 pipeline 的模拟输入。

> 抽样逻辑与 `run_20260730_background.py` 完全一致，便于在 notebook 中交互调试。
> 若数据量很大，仍建议用后台脚本批量运行。

In [ ]:
# ============================================================
# 0) 导入与基础路径
# ============================================================
import sys
import os
import gc
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# 自动定位项目根（TPC_DE_SIm-test）：兼容 Jupyter 在 notebooks/ 或项目根下启动
_HERE = Path.cwd().resolve()
_PROJ_ROOT = _HERE
while _PROJ_ROOT.name != "TPC_DE_SIm-test" and _PROJ_ROOT.parent != _PROJ_ROOT:
    _PROJ_ROOT = _PROJ_ROOT.parent
if _PROJ_ROOT.name != "TPC_DE_SIm-test":
    raise RuntimeError("未找到项目根目录 TPC_DE_SIm-test，请在项目内启动 Jupyter。")
print("项目根 _PROJ_ROOT =", _PROJ_ROOT)

if str(_PROJ_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJ_ROOT))

from relics_de_sim.config import DESimConfig
from relics_de_sim.pipeline import Pipeline
from relics_de_sim.cuts import pattern_likelihood
print("relics_de_sim imported")


In [ ]:
# ============================================================
# 1) 抽样参数设置（按需修改）
# ============================================================
SIM_TIME   = 10000.0   # 模拟总时长 [s]
MUON_RATE  = 6.0       # muon 事件率 [Hz]  -> muon 目标数 = SIM_TIME * MUON_RATE
GAMMA_RATE = 1.0       # gamma 事件率 [Hz] -> gamma 目标数 = SIM_TIME * GAMMA_RATE

N_MUON_FILES  = 1      # 抽样样本：muon_events 文件个数
N_GAMMA_FILES = 6      # 抽样样本：gamma_events 文件个数
START_INDEX   = 1      # 样本文件起始编号（muon_events.<i>.npy / gamma_events.<i>.npy）

SEED  = 7              # 随机种子（None 表示不固定）
DEVICE = "auto"        # 位置重建 CNN 设备："auto" / "cuda" / "cpu"

# 合并后的输入数组写到哪里（作为两个 pipeline 的输入）
MERGED_INPUT = _PROJ_ROOT / "output" / "20260730_sampled" / "merged_input.npy"
MERGED_INPUT.parent.mkdir(parents=True, exist_ok=True)

N_MUON_TARGET  = int(round(SIM_TIME * MUON_RATE))    # 如 10000*6 = 60000
N_GAMMA_TARGET = int(round(SIM_TIME * GAMMA_RATE))   # 如 10000*1 = 10000

print(f"模拟总时长: {SIM_TIME} s")
print(f"muon  事件率: {MUON_RATE} Hz -> 目标 {N_MUON_TARGET} 个 eventId")
print(f"gamma 事件率: {GAMMA_RATE} Hz -> 目标 {N_GAMMA_TARGET} 个 eventId")
print(f"抽样样本文件: muon {N_MUON_FILES} 个, gamma {N_GAMMA_FILES} 个 (起始编号 {START_INDEX})")
print("合并输入将保存到:", MERGED_INPUT)


In [ ]:
# ============================================================
# 2) 抽样函数（与 scripts/run_20260730_background.py 一致）
# ============================================================
def collect_event_files(base_dir, event_type):
    """查找 *_events 文件：优先 <base>/<type>_events/<type>_events.*.npy，回退 *.npy。"""
    base = Path(base_dir)
    event_dir = base / f"{event_type}_events"
    matches = sorted(event_dir.glob(f"{event_type}_events.*.npy")) if event_dir.exists() else []
    if not matches:
        matches = sorted(base.rglob(f"{event_type}_events.*.npy"))
    if not matches:
        matches = sorted(base.rglob("*.npy"))
    return matches


def load_events_from_files(files):
    """加载多个 *_events 文件并合并，同时把 eventId 偏移成全局唯一。"""
    arrays = []
    pre_event_num = 0
    for path in files:
        arr = np.load(path)
        file_event_num = int(arr["eventId"][-1]) + 1
        arr = arr.copy()
        arr["eventId"] = arr["eventId"] + pre_event_num
        pre_event_num += file_event_num
        arrays.append(arr)
    return np.concatenate(arrays, axis=0)


def sample_events_by_eventid(merged, n_target, rng, event_type):
    """按 eventId 抽样到 n_target 个事件。

    1) 优先随机无放回；
    2) 候选不足时先无放回抽全部，再有放回补足；
    3) 重复抽到的事件分配一个全新的、不重复的 eventId。
    """
    unique_ids = np.unique(merged["eventId"])
    n_avail = len(unique_ids)

    if n_avail >= n_target:
        chosen = rng.choice(unique_ids, size=n_target, replace=False)
        picked = merged[np.isin(merged["eventId"], chosen)]
        print(f"{event_type}: 无放回抽样 {n_target} 个 eventId，共 {len(picked)} 行（候选 {n_avail}）")
        return picked

    print(f"{event_type}: 候选 eventId 数 {n_avail} < 目标 {n_target}，先无放回抽全部，再有放回补足")
    base_part = merged[np.isin(merged["eventId"], unique_ids)]

    need = n_target - n_avail
    repeat_ids = rng.choice(unique_ids, size=need, replace=True)
    next_id = int(merged["eventId"].max()) + 1
    extra_parts = []
    for eid in repeat_ids:
        sub = merged[merged["eventId"] == eid].copy()
        sub["eventId"] = next_id
        next_id += 1
        extra_parts.append(sub)

    picked = np.concatenate([base_part] + extra_parts, axis=0) if extra_parts else base_part
    print(f"{event_type}: 抽样 {n_target} 个 eventId（无放回 {n_avail} + 有放回补足 {need}），共 {len(picked)} 行")
    return picked


def merge_muon_gamma(muon_arr, gamma_arr):
    """合并 muon 与 gamma，并把 eventId 从 0 重新排序。

    关键：先把 gamma 的 eventId 偏移到 muon 最大值之上，避免两类事件编号重叠被误并。
    """
    offset = int(muon_arr["eventId"].max()) + 1
    gamma_arr2 = gamma_arr.copy()
    gamma_arr2["eventId"] = gamma_arr2["eventId"] + offset

    merged = np.concatenate([muon_arr, gamma_arr2], axis=0)
    merged = merged[np.argsort(merged["eventId"], kind="stable")]
    merged["eventId"] = np.unique(merged["eventId"], return_inverse=True)[1]
    print(f"合并后共 {len(merged)} 行，{int(merged['eventId'][-1]) + 1} 个不同 eventId")
    return merged


In [ ]:
# ============================================================
# 3) 执行抽样：加载样本文件 -> 按 eventId 抽样 -> 合并 -> 保存合并输入
# ============================================================
muon_root = _PROJ_ROOT / "muon_track"
muon_candidates = collect_event_files(muon_root, "muon")
gamma_candidates = collect_event_files(muon_root, "gamma")
print(f"可用样本文件: muon={len(muon_candidates)}, gamma={len(gamma_candidates)}")
if not muon_candidates or not gamma_candidates:
    raise FileNotFoundError("未找到 muon_events 或 gamma_events 样本文件。")

n_c_muon, n_c_gamma = len(muon_candidates), len(gamma_candidates)

# 选取样本文件：muon 按 START_INDEX 起连续 N_MUON_FILES 个；gamma 起连续 N_GAMMA_FILES 个
muon_start = START_INDEX - 1
muon_sample = [muon_candidates[(muon_start + j) % n_c_muon] for j in range(N_MUON_FILES)]
gamma_sample = [gamma_candidates[(muon_start + j) % n_c_gamma] for j in range(N_GAMMA_FILES)]
print("  样本 muon :", [p.name for p in muon_sample])
print("  样本 gamma:", [p.name for p in gamma_sample])

rng = np.random.default_rng(SEED)

# 加载样本并抽样
muon_loaded = load_events_from_files(muon_sample)
gamma_loaded = load_events_from_files(gamma_sample)
muon_sampled = sample_events_by_eventid(muon_loaded, N_MUON_TARGET, rng, "muon")
gamma_sampled = sample_events_by_eventid(gamma_loaded, N_GAMMA_TARGET, rng, "gamma")

# 合并（eventId 从 0 重排）并保存为合并输入数组
merged_input = merge_muon_gamma(muon_sampled, gamma_sampled)
np.save(MERGED_INPUT, merged_input)
print("合并输入已保存:", MERGED_INPUT, "| shape =", merged_input.shape)
print("dtype:", merged_input.dtype)

# 释放抽样中间量
del muon_loaded, gamma_loaded, muon_sampled, gamma_sampled
gc.collect()


In [ ]:
# ============================================================
# 4) DE pipeline：用抽样合并数组作为输入
# ============================================================
cfg = DESimConfig.from_yaml(_PROJ_ROOT / 'configs' / 'muon_only.yaml')
print('Loaded YAML source:', cfg.source_yaml)
print(' muon_track_dir:', cfg.paths.muon_track_dir)
print(' dead_time_s =', cfg.simulation.dead_time_s)

# 用抽样合并数组替代原直接导入的 muon/gamma 文件
cfg.paths.muon_track_dir = str(muon_root)
cfg.simulation.muon_rate_hz = MUON_RATE + GAMMA_RATE   # 总事件率

pipe = Pipeline(cfg, rng=np.random.default_rng(SEED), device=DEVICE)
print("Pipeline created. device =", pipe.device)

pipe.load_muon_tracks([str(MERGED_INPUT)])
print("Loaded muons: merged shape =", None if pipe.merged_muon is None else pipe.merged_muon.shape)
print("dense muon shape =", None if pipe.dense_muon is None else pipe.dense_muon.shape)
print("time range [s] =", pipe.time_range_s)


In [ ]:
# ============================================================
# 5) DE 仿真：延迟电子 -> 堆叠 pattern -> 位置重建 -> 评分
# ============================================================
if pipe.dense_muon is None:
    print('未生成 dense_muon —— 请先运行上一步')
else:
    print('simulate_delayed_electrons() ...')
    pipe.simulate_delayed_electrons()
    print('simulate_pile_up_patterns() ...')
    pipe.simulate_pile_up_patterns()
    print('recon_position_de() ...')
    try:
        pipe.recon_position_de()
    except Exception as e:
        print('位置重建失败（可能缺少模型或 PyTorch），错误：', e)
    print('score() ...')
    try:
        pipe.score()
    except Exception as e:
        print('评分失败：', e)
    for order, res in zip(pipe.pile_up_orders, pipe.pile_up_results):
        print(f'order={order} kept={len(res.pe_by_area)}')

# DE 侧关键数组（与后续分析/画图 cell 的变量名保持一致）
pattern = np.concatenate(pipe.pile_up_pattern_coef)
st_cor = np.concatenate(pipe.pile_up_st_cor)

DE_pe_by_area = np.concatenate([r.pe_by_area for r in pipe.pile_up_results])
DE_area = np.sum(DE_pe_by_area, axis=1)
print("DE: pattern=%d  st_cor=%d  area=%d" % (len(pattern), len(st_cor), len(DE_area)))


In [ ]:
# ============================================================
# 6) CEvNS pipeline：用同一抽样合并数组作为输入
# ============================================================
cfg_cevns = DESimConfig.from_yaml(_PROJ_ROOT / 'configs' / 'cevns_sim.yaml')
print('cevns_e_spectrum:', cfg_cevns.paths.cevns_e_spectrum)
cfg_cevns.paths.muon_track_dir = str(muon_root)

pipe_cevns = Pipeline(cfg_cevns, rng=np.random.default_rng(SEED), device=DEVICE)
pipe_cevns.load_muon_tracks([str(MERGED_INPUT)])
print('CEvNS time range [s] =', pipe_cevns.time_range_s)

try:
    pipe_cevns.simulate_cevns()
    for arr in pipe_cevns.cevns_points:
        if len(arr):
            print(f'n_e={int(arr["num_e"][0])}: {len(arr)} events')
except ImportError as exc:
    print('CNN inference unavailable:', exc)

# CEvNS 侧关键数组（与后续分析/画图 cell 的变量名保持一致）
arr = np.concatenate(pipe_cevns.cevns_points)
cevns_st_cor = arr['st_cor']
cevns_pattern = pattern_likelihood(arr['pe_by_area'], arr['recons_light_pattern'], n_top=28)
cevns_area = np.sum(arr['pe_by_area'], axis=1)
cevns_pe_by_area = arr['pe_by_area']
print("CEvNS: st_cor=%d  pattern=%d  area=%d" % (len(cevns_st_cor), len(cevns_pattern), len(cevns_area)))


## 7) 后续分析

到这里，以下变量已就绪，**可直接复用原 `20260730.ipynb` 中的分析与画图 cell**：

| 变量 | 含义 |
|---|---|
| `st_cor`, `pattern`, `DE_area` | DE 堆叠信号的 space-time correlation / pattern likelihood / area |
| `cevns_st_cor`, `cevns_pattern`, `cevns_area` | CEvNS 信号对应量 |
| `DE_pe_by_area`, `cevns_pe_by_area` | 各事件各通道 PE（用于 `pattern_plot`） |
| `pipe`, `pipe_cevns` | 两个 pipeline 对象（含 `pmt_top` 等） |

下面给出一个最小验证图（`log(st_cor) vs pattern`）。

In [ ]:
# ============================================================
# 8) 验证图：DE 与 CEvNS 的 log(st_cor) vs pattern likelihood
# ============================================================
from matplotlib.colors import LogNorm

x_bins = np.arange(-10, 25, 0.1)
y_bins = np.arange(-250, 0, 1)

fig, ax = plt.subplots(figsize=(6, 5))
h = ax.hist2d(np.log(cevns_st_cor), cevns_pattern, bins=(x_bins, y_bins),
              cmap='cool', norm=LogNorm(), alpha=0.7)
h = ax.hist2d(np.log(st_cor), pattern, bins=(x_bins, y_bins),
              cmap='viridis', norm=LogNorm(), alpha=0.7)
ax.set_xlabel('log(st_cor)')
ax.set_ylabel('pattern likelihood')
plt.colorbar(h[3], ax=ax)
plt.xlim(-10, 25)
plt.ylim(-250, 0)
plt.show()

# pattern_plot（复用原 notebook 的绘图函数，pmt 布局来自 pipe.pmt_top）
def pattern_plot(phe_pmt, electron=[], pos_recons=[], pmt_info=None, title="", savefig=False):
    import matplotlib as mpl
    import matplotlib.cm as cm
    from matplotlib.patches import Rectangle
    from mpl_toolkits.axes_grid1 import ImageGrid

    if pmt_info is None:
        pmt_info = pipe.pmt_top
    phe_pmt = np.nan_to_num(np.asarray(phe_pmt), nan=0.0)
    max_val = phe_pmt.max()
    norm = (mpl.colors.Normalize(vmin=0, vmax=max(1, max_val)) if max_val <= 1
            else mpl.colors.LogNorm(vmin=1, vmax=max_val, clip=True))
    cmap = mpl.cm.viridis

    fig = plt.figure(figsize=(7, 6))
    grid = ImageGrid(fig, 111, (1, 1))
    for pmt_id in range(28):
        pmt = pmt_info[pmt_id]
        color = 'white' if phe_pmt[pmt_id] < 0.1 else cmap(norm(phe_pmt[pmt_id]))
        grid[0].text(pmt['x'], pmt['y'], str(round(phe_pmt[pmt_id], 1)),
                     ha='center', va='center', fontsize=15)
        grid[0].add_patch(Rectangle(xy=(pmt['x'] - 1.27, pmt['y'] - 1.27), width=2.54,
                                    height=2.54, angle=pmt['rot_z'], rotation_point='center',
                                    fc='none', ec='black', lw=1))
        grid[0].add_patch(Rectangle(xy=(pmt['x'] - 1.27, pmt['y'] - 1.27), width=2.54,
                                    height=2.54, angle=pmt['rot_z'], rotation_point='center',
                                    fc=color, alpha=1))
    grid[0].axis('equal')
    grid[0].set_xlabel('x/cm')
    grid[0].set_ylabel('y/cm')
    grid[0].set_title(title)
    if savefig:
        plt.savefig(title + ".png")
    plt.show()
    return 0

# 示例：DE 前 3 个事件的 PMT 光斑
for i in range(min(3, len(DE_area))):
    pattern_plot(DE_pe_by_area[i], pmt_info=pipe.pmt_top,
                 title=f"DE PE by area (event {i}, area={DE_area[i]:.0f})")
